In [42]:
import json

import pandas as pd
import ast
import os
from tqdm import tqdm

from typing import Union, List

In [43]:
tqdm.pandas()

# Load Impact DF

In [44]:
def try_literal_eval(x):
    if isinstance(x, str):
        x = x.strip()
        if (x.startswith("[") and x.endswith("]")) or \
           (x.startswith("{") and x.endswith("}")) or \
           (x.startswith("(") and x.endswith(")")):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return x
    return x

In [45]:
dataset_dir = "/home/yishin/keith/patent_research/model_io"

In [46]:
Impact_df = pd.read_csv(os.path.join(dataset_dir, "Base.csv"), encoding="utf-8")
Impact_df = Impact_df.map(try_literal_eval)

Impact_df.head()

,title,caption,file_names,fig_desc,class
0,Data reader,"The image is a 3D drawing of a data reader, wh...",[impact_dataset/2022/USD0949851-20220426/USD09...,"[FIG. 1 is a front, left-side, top perspective...","D14357,D14358"
1,Panel light,"The image is a white, rectangular shape, which...",[impact_dataset/2022/USD0971479-20221129/USD09...,[FIG. 1 is a perspective view of the panel lig...,D26 74
2,Massager,"The image is a white outline of a massager, wh...",[impact_dataset/2022/USD0959008-20220726/USD09...,[FIG. 1 is a first perspective view of a massa...,D24215
3,Luggage,"The image is a square-shaped suitcase, which i...",[impact_dataset/2022/USD0965975-20221011/USD09...,[FIG. 1 shows a perspective view of a luggage ...,"D 3279, D3273"
4,Wall-mounted safe,The image is a white drawing of a wall-mounted...,[impact_dataset/2022/USD0942735-20220201/USD09...,[FIG. 1 is a perspective view of a wall-mounte...,D99 28


# Preprocessing

## USPC/Locarno

### USPC Normalization

In [47]:
with open("class_folder/USPC_RANGES.json", "r", encoding="utf-8") as f:
    USPC_RANGES = json.load(f)

In [48]:
def normalize_code(code: Union[str, List[str]]) -> set[str]:
    results = set()

    # ---- normalize input into a list of strings ----
    if isinstance(code, str):
        codes_to_process = code.replace(" ", "").split(",")
    else:
        # list of strings → clean each, split commas if present
        codes_to_process = []
        for c in code:
            if isinstance(c, str):
                codes_to_process.extend(c.replace(" ", "").split(","))

    # ---- core normalization logic ----
    for single_code in codes_to_process:
        for prefix_len in (2, 3):
            if len(single_code) <= prefix_len:
                continue

            prefix = single_code[:prefix_len]
            suffix = single_code[prefix_len:]

            # reject leading-zero suffixes
            if len(suffix) > 1 and suffix.startswith("0"):
                continue

            ranges = USPC_RANGES.get(prefix)
            if not ranges:
                continue

            try:
                suffix_int = int(suffix)
            except ValueError:
                continue

            start, end = ranges
            if start <= suffix_int < end:
                results.add(f"{prefix}-{suffix}")

    return results

In [49]:
Impact_df["USPC_class"] = Impact_df["class"].apply(normalize_code)

Impact_df.head()

,title,caption,file_names,fig_desc,class,USPC_class
0,Data reader,"The image is a 3D drawing of a data reader, wh...",[impact_dataset/2022/USD0949851-20220426/USD09...,"[FIG. 1 is a front, left-side, top perspective...","D14357,D14358","{D14-357, D14-358}"
1,Panel light,"The image is a white, rectangular shape, which...",[impact_dataset/2022/USD0971479-20221129/USD09...,[FIG. 1 is a perspective view of the panel lig...,D26 74,"{D2-674, D26-74}"
2,Massager,"The image is a white outline of a massager, wh...",[impact_dataset/2022/USD0959008-20220726/USD09...,[FIG. 1 is a first perspective view of a massa...,D24215,{D24-215}
3,Luggage,"The image is a square-shaped suitcase, which i...",[impact_dataset/2022/USD0965975-20221011/USD09...,[FIG. 1 shows a perspective view of a luggage ...,"D 3279, D3273","{D32-73, D3-279, D3-273}"
4,Wall-mounted safe,The image is a white drawing of a wall-mounted...,[impact_dataset/2022/USD0942735-20220201/USD09...,[FIG. 1 is a perspective view of a wall-mounte...,D99 28,{D99-28}


### USPC to Locarno Conversion

In [50]:
with open("class_folder/USPC_CONVERSION_CHART.json", "r", encoding="utf-8") as f:
    CONVERSION_CHART = json.load(f)

In [51]:
def convert_USPC_Locarno(USPC_Codes: set):
    Locarno_Codes = []

    for i in USPC_Codes:
        try:
            main_class, subclass = i.split("-")
        except ValueError:
            return set()
        
        for cat in CONVERSION_CHART[main_class]:
            bound = cat["U.S. Subclass"].replace(" ", "").split("-")

            if len(bound) == 1:
                if subclass == bound[0]:
                    Locarno_Codes.append(cat["Locarno Class - Subclass"].replace(" ", ""))
            else:
                if float(bound[0]) <= float(subclass) <= float(bound[1]):
                    Locarno_Codes.append(cat["Locarno Class - Subclass"].replace(" ", ""))

    return set(Locarno_Codes)

In [52]:
Impact_df["Loc_class"] = Impact_df["USPC_class"].apply(convert_USPC_Locarno)

Impact_df.head()

,title,caption,file_names,fig_desc,class,USPC_class,Loc_class
0,Data reader,"The image is a 3D drawing of a data reader, wh...",[impact_dataset/2022/USD0949851-20220426/USD09...,"[FIG. 1 is a front, left-side, top perspective...","D14357,D14358","{D14-357, D14-358}",{14-02}
1,Panel light,"The image is a white, rectangular shape, which...",[impact_dataset/2022/USD0971479-20221129/USD09...,[FIG. 1 is a perspective view of the panel lig...,D26 74,"{D2-674, D26-74}",{26-05}
2,Massager,"The image is a white outline of a massager, wh...",[impact_dataset/2022/USD0959008-20220726/USD09...,[FIG. 1 is a first perspective view of a massa...,D24215,{D24-215},{24-01}
3,Luggage,"The image is a square-shaped suitcase, which i...",[impact_dataset/2022/USD0965975-20221011/USD09...,[FIG. 1 shows a perspective view of a luggage ...,"D 3279, D3273","{D32-73, D3-279, D3-273}","{03-01, 07-05}"
4,Wall-mounted safe,The image is a white drawing of a wall-mounted...,[impact_dataset/2022/USD0942735-20220201/USD09...,[FIG. 1 is a perspective view of a wall-mounte...,D99 28,{D99-28},"{25-02, 06-04}"


### Drop USPC

In [53]:
Impact_df = Impact_df.drop(columns=["class"])
Impact_df = Impact_df.drop(columns=["USPC_class"])

### Empty Check

In [54]:
def is_class_empty(x):
    return (
        pd.isna(x)
        or isinstance(x, str)
        or (isinstance(x, set) and len(x) == 0)
    )

In [55]:
mask = (
    Impact_df["Loc_class"].apply(is_class_empty)
)
Impact_df = Impact_df[~mask]

In [56]:
def to_set_or_empty(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {}
    if isinstance(x, (list, tuple, set)):
        return set(x)
    return {str(x)}

In [57]:
row_has_empty_mask = Impact_df.apply(
    lambda row: any(len(to_set_or_empty(v)) == 0 for v in row),
    axis=1,
)

In [58]:
row_has_empty_mask.sum()
Impact_df = Impact_df[~row_has_empty_mask].reset_index(drop=True)
Impact_df.shape[0]

30868

In [59]:
Impact_df.head()

,title,caption,file_names,fig_desc,Loc_class
0,Data reader,"The image is a 3D drawing of a data reader, wh...",[impact_dataset/2022/USD0949851-20220426/USD09...,"[FIG. 1 is a front, left-side, top perspective...",{14-02}
1,Panel light,"The image is a white, rectangular shape, which...",[impact_dataset/2022/USD0971479-20221129/USD09...,[FIG. 1 is a perspective view of the panel lig...,{26-05}
2,Massager,"The image is a white outline of a massager, wh...",[impact_dataset/2022/USD0959008-20220726/USD09...,[FIG. 1 is a first perspective view of a massa...,{24-01}
3,Luggage,"The image is a square-shaped suitcase, which i...",[impact_dataset/2022/USD0965975-20221011/USD09...,[FIG. 1 shows a perspective view of a luggage ...,"{03-01, 07-05}"
4,Wall-mounted safe,The image is a white drawing of a wall-mounted...,[impact_dataset/2022/USD0942735-20220201/USD09...,[FIG. 1 is a perspective view of a wall-mounte...,"{25-02, 06-04}"


## Class Dissection

In [60]:
def split_class(value):
    if not value:
        return pd.Series([None, None])

    first_item = list(value)[0]

    main_class, sub_class = first_item.split("-")

    return pd.Series([main_class, sub_class])

In [61]:
Impact_df[["main_class", "sub_class"]] = (
    Impact_df["Loc_class"].apply(split_class)
)

In [62]:
Impact_df.head(10)

,title,caption,file_names,fig_desc,Loc_class,main_class,sub_class
0,Data reader,"The image is a 3D drawing of a data reader, wh...",[impact_dataset/2022/USD0949851-20220426/USD09...,"[FIG. 1 is a front, left-side, top perspective...",{14-02},14,02
1,Panel light,"The image is a white, rectangular shape, which...",[impact_dataset/2022/USD0971479-20221129/USD09...,[FIG. 1 is a perspective view of the panel lig...,{26-05},26,05
2,Massager,"The image is a white outline of a massager, wh...",[impact_dataset/2022/USD0959008-20220726/USD09...,[FIG. 1 is a first perspective view of a massa...,{24-01},24,01
3,Luggage,"The image is a square-shaped suitcase, which i...",[impact_dataset/2022/USD0965975-20221011/USD09...,[FIG. 1 shows a perspective view of a luggage ...,"{03-01, 07-05}",03,01
4,Wall-mounted safe,The image is a white drawing of a wall-mounted...,[impact_dataset/2022/USD0942735-20220201/USD09...,[FIG. 1 is a perspective view of a wall-mounte...,"{25-02, 06-04}",25,02
5,Pair of earrings,The image is a white outline of a pair of earr...,[impact_dataset/2022/USD0964203-20220920/USD09...,"[1.1 : Front, 1.2 : Back, 1.3 : Left, 1.4 : Ri...",{11-01},11,01
6,Razor hanger,The image is a white drawing of a razor hanger...,[impact_dataset/2022/USD0964065-20220920/USD09...,[FIG. 1 is a first perspective view of a desig...,{06-04},06,04
7,Toy,"The image is a drawing of a toy giraffe, and i...",[impact_dataset/2022/USD0942553-20220201/USD09...,[FIG. 1 is a perspective view of a toy showing...,{21-01},21,01
8,Single shot protection device,The image is a black and white drawing of a si...,[impact_dataset/2022/USD0947978-20220405/USD09...,[FIG. 1 is a front perspective view of a singl...,{22-01},22,01
9,Pet feeding station,The image is a black and white drawing of a pe...,[impact_dataset/2022/USD0970824-20221122/USD09...,[FIG. 1 is a perspective view of a pet feeding...,{30-03},30,03


## Image Paths and Fig Desc

In [63]:
from __future__ import annotations
 
import re
from typing import Optional

In [64]:
Impact_df = Impact_df.rename(columns={"file_names": "image_paths"})

Impact_df.head()

,title,caption,image_paths,fig_desc,Loc_class,main_class,sub_class
0,Data reader,"The image is a 3D drawing of a data reader, wh...",[impact_dataset/2022/USD0949851-20220426/USD09...,"[FIG. 1 is a front, left-side, top perspective...",{14-02},14,02
1,Panel light,"The image is a white, rectangular shape, which...",[impact_dataset/2022/USD0971479-20221129/USD09...,[FIG. 1 is a perspective view of the panel lig...,{26-05},26,05
2,Massager,"The image is a white outline of a massager, wh...",[impact_dataset/2022/USD0959008-20220726/USD09...,[FIG. 1 is a first perspective view of a massa...,{24-01},24,01
3,Luggage,"The image is a square-shaped suitcase, which i...",[impact_dataset/2022/USD0965975-20221011/USD09...,[FIG. 1 shows a perspective view of a luggage ...,"{03-01, 07-05}",03,01
4,Wall-mounted safe,The image is a white drawing of a wall-mounted...,[impact_dataset/2022/USD0942735-20220201/USD09...,[FIG. 1 is a perspective view of a wall-mounte...,"{25-02, 06-04}",25,02


In [65]:
_EXCLUDE_PHRASES = [
    r"reference view",
    r"state of use",
    r"state of conduction",
    r"in use",
    r"environment",
    r"prior art",
    r"exploded view",
    r"cross.?section",
    r"sectional view",
    r"detail view",
    r"enlarged view",
    r"schematic",
]

_EXCLUDE_RE = re.compile(
    "|".join(_EXCLUDE_PHRASES),
    re.IGNORECASE,
)

In [66]:
_TIER: list[tuple[int, re.Pattern]] = [
    # Perspective views capture three faces at once — highest value.
    (100, re.compile(r"perspective", re.IGNORECASE)),
 
    # Primary faces: front and rear.
    (70,  re.compile(r"\bfront\b", re.IGNORECASE)),
    (65,  re.compile(r"\brear\b",  re.IGNORECASE)),
 
    # Secondary faces: sides.
    (50,  re.compile(r"\b(left|right).?side\b", re.IGNORECASE)),
    (45,  re.compile(r"\bside\b",  re.IGNORECASE)),
 
    # Plan views — useful but often less distinctive.
    (30,  re.compile(r"\btop\b",    re.IGNORECASE)),
    (25,  re.compile(r"\bbottom\b", re.IGNORECASE)),
]

_MULTI_FACE_BONUS = 15   # added when a description mentions 2+ anatomical terms
_FACE_TERMS = re.compile(
    r"\b(front|rear|left|right|top|bottom|side)\b",
    re.IGNORECASE,
)

In [67]:
def _score(description: str) -> int:
    """Return an importance score for a single figure description."""
    best = 0
    for score, pattern in _TIER:
        if pattern.search(description):
            best = max(best, score)
 
    # Bonus when the view description mentions multiple spatial terms
    # (e.g. "front, left-side, top perspective").
    face_count = len(_FACE_TERMS.findall(description))
    if face_count >= 2:
        best += _MULTI_FACE_BONUS
 
    return best

def _parse_fig_number(description: str) -> int:
    """Extract the FIG number for stable tie-breaking (lower = earlier)."""
    match = re.search(r"\bFIG\.?\s*(\d+)", description, re.IGNORECASE)
    return int(match.group(1)) if match else 9999
 
 
def select_important_figures(
    descriptions: list[str],
    n: int = 4,
    exclude_re: Optional[re.Pattern] = None,
) -> list[dict]:
    """
    Select the *n* most informative figure descriptions.
 
    Parameters
    ----------
    descriptions : list[str]
        Raw figure-description strings, e.g. from the patent's brief
        description of drawings section.
 
    n : int
        Number of figures to return (default 4).
 
    exclude_re : re.Pattern, optional
        Override the default exclusion regex.  Pass ``None`` to use the
        built-in ``_EXCLUDE_RE``.
 
    Returns
    -------
    list[dict]
        Sorted by score descending, each entry::
 
            {
                "fig":         "FIG. 1",
                "description": "FIG. 1 is a front, left-side, top ...",
                "score":       115,
            }
 
    Examples
    --------
    >>> results = select_important_figures(descriptions, n=3)
    >>> for r in results:
    ...     print(r["fig"], r["score"], r["description"][:60])
    """
    excl = exclude_re if exclude_re is not None else _EXCLUDE_RE
 
    scored = []
    for desc in descriptions:
        if excl.search(desc):
            continue                        # skip reference / non-ornamental views
        score = _score(desc)
        if score == 0:
            continue                        # unrecognised / unlabelled — skip
        scored.append({
            "fig":         re.search(r"FIG\.?\s*\d+", desc, re.IGNORECASE).group() if re.search(r"FIG\.?\s*\d+", desc, re.IGNORECASE) else "?",
            "description": desc,
            "score":       score,
            "_fig_num":    _parse_fig_number(desc),
        })
 
    # Sort: score descending, then fig number ascending for stable tie-breaking.
    scored.sort(key=lambda x: (-x["score"], x["_fig_num"]))
 
    top = scored[:n]
 
    # Return without the internal sort key.
    return [{k: v for k, v in entry.items() if k != "_fig_num"} for top in [top] for entry in top]
 
 
# ---------------------------------------------------------------------------
# Convenience wrapper for use inside a DataFrame pipeline
# ---------------------------------------------------------------------------
 
def select_fig_indices(descriptions: list[str], n: int = 4) -> list[int]:
    """
    Return the 0-based indices into *descriptions* of the selected figures.
 
    Useful when you need to filter a parallel list of image paths::
 
        fig_descs  = row["fig_descriptions"]   # list[str]
        image_paths = row["image_paths"]        # list[str], same order
 
        indices     = select_fig_indices(fig_descs, n=4)
        top_paths   = [image_paths[i] for i in indices]
    """
    selected_descs = {
        entry["description"]
        for entry in select_important_figures(descriptions, n=n)
    }
    return [i for i, d in enumerate(descriptions) if d in selected_descs]

In [68]:
Impact_df["best_fig_desc"] = (
    Impact_df["fig_desc"]
    .map(try_literal_eval)
    .apply(lambda descs: {
        r["fig"]: r["description"]
        for r in select_important_figures(descs, n=4)
    })
)

In [69]:
Impact_df["best_fig_desc"].iloc[0]

{'FIG. 1': 'FIG. 1 is a front, left-side, top perspective view of the data reader;',
 'FIG. 8': 'FIG. 8 is a rear, right-side, bottom perspective view thereof;',
 'FIG. 2': 'FIG. 2 is a front elevational view thereof;',
 'FIG. 3': 'FIG. 3 is a rear elevational view thereof;'}

# Subsampling

In [70]:
len(Impact_df)

30868

In [71]:
print(Impact_df.index.name)
print(Impact_df.columns.tolist())

None
['title', 'caption', 'image_paths', 'fig_desc', 'Loc_class', 'main_class', 'sub_class', 'best_fig_desc']


In [72]:
def sample_group(group, frac=0.1, min_n=5):
    n = max(int(len(group) * frac), min_n)
    n = min(n, len(group))
    return group.sample(n=n, random_state=42)

sample_df = (
    Impact_df
    .groupby("main_class", group_keys=True)
    .apply(sample_group, frac=0.1, min_n=5)
    .reset_index(level=0)
    .reset_index(drop=True)
)

sample_df.head(10)

,main_class,title,caption,image_paths,fig_desc,Loc_class,sub_class,best_fig_desc
0,01,Cupcake with contrasting icing,The image is a black and white picture of a cu...,[impact_dataset/2022/USD0957089-20220712/USD09...,[FIG. 1 is a perspective view of a cupcake wit...,"{01-01, 11-01}",01,{'FIG. 1': 'FIG. 1 is a perspective view of a ...
1,01,Ring with an integrated spoon,The image is a drawing of a ring with an integ...,[impact_dataset/2022/USD0966130-20221011/USD09...,[FIG. 1 is a front perspective view of a ring ...,"{01-01, 11-01}",01,{'FIG. 1': 'FIG. 1 is a front perspective view...
2,01,Butter stick,The image is a white and black drawing of a bu...,[impact_dataset/2022/USD0962586-20220906/USD09...,"[FIG. 1 is a front, top and left side perspect...","{01-06, 11-01}",06,"{'FIG. 1': 'FIG. 1 is a front, top and left si..."
3,01,Necklace,"The image is a round shape, and the necklace i...",[impact_dataset/2022/USD0962109-20220830/USD09...,[FIG. 1 is a front view of a necklace of the p...,"{01-01, 11-01}",01,{'FIG. 3': 'FIG. 3 is a right side perspective...
4,01,Dietary supplement,"The image is circular in shape, and it represe...",[impact_dataset/2022/USD0941457-20220118/USD09...,[FIG. 1 is a top perspective view of a dietary...,"{01-01, 11-01}",01,{'FIG. 1': 'FIG. 1 is a top perspective view o...
5,01,Rolled pet treat,"The image is a rolled pet treat, which is a ty...",[impact_dataset/2022/USD0973298-20221227/USD09...,"[FIG. 1 is top, front and left side perspectiv...",{01-01},01,"{'FIG. 1': 'FIG. 1 is top, front and left side..."
6,01,Sandwich,The image is a black and white drawing of a sa...,[impact_dataset/2022/USD0957086-20220712/USD09...,"[FIG. 1 is a top, rear, left side perspective ...","{01-01, 11-01}",01,"{'FIG. 1': 'FIG. 1 is a top, rear, left side p..."
7,01,Combined ice cream cone and ice cream cone por...,"The image is a white triangle, which is the sh...",[impact_dataset/2022/USD0948161-20220412/USD09...,[FIG. 1 is a front elevation view of a combine...,"{01-01, 11-01}",01,{'FIG. 5': 'FIG. 5 is a bottom perspective vie...
8,01,Food product,"The image is a square-shaped food product, whi...",[impact_dataset/2022/USD0968749-20221108/USD09...,[FIG. 1 is a front elevational view of a food ...,"{01-01, 11-01}",01,{'FIG. 7': 'FIG. 7 is a front perspective view...
9,01,Confectionery,"The image is a white, cylindrical shape, which...",[impact_dataset/2022/USD0950191-20220503/USD09...,"[1. Confectionery, 1.1 : Angled Side/Top View,...","{01-01, 11-01}",01,{'?': '1.3 : Bottom'}


In [73]:
len(sample_df)

3074

In [74]:
print(sample_df["main_class"].value_counts())

main_class
14    471
12    246
24    241
02    228
23    195
21    166
09    142
06    141
08    141
13    141
15    116
07    105
10     86
26     83
16     72
11     63
28     63
25     61
03     56
30     43
04     37
27     36
22     30
19     26
29     25
18     16
01     12
20     11
31      9
17      7
05      5
Name: count, dtype: int64


In [75]:
sample_df = sample_df[["title", "caption", "image_paths", "best_fig_desc", "Loc_class", "main_class", "sub_class"]]

In [76]:
sample_df.head()

,title,caption,image_paths,best_fig_desc,Loc_class,main_class,sub_class
0,Cupcake with contrasting icing,The image is a black and white picture of a cu...,[impact_dataset/2022/USD0957089-20220712/USD09...,{'FIG. 1': 'FIG. 1 is a perspective view of a ...,"{01-01, 11-01}",01,01
1,Ring with an integrated spoon,The image is a drawing of a ring with an integ...,[impact_dataset/2022/USD0966130-20221011/USD09...,{'FIG. 1': 'FIG. 1 is a front perspective view...,"{01-01, 11-01}",01,01
2,Butter stick,The image is a white and black drawing of a bu...,[impact_dataset/2022/USD0962586-20220906/USD09...,"{'FIG. 1': 'FIG. 1 is a front, top and left si...","{01-06, 11-01}",01,06
3,Necklace,"The image is a round shape, and the necklace i...",[impact_dataset/2022/USD0962109-20220830/USD09...,{'FIG. 3': 'FIG. 3 is a right side perspective...,"{01-01, 11-01}",01,01
4,Dietary supplement,"The image is circular in shape, and it represe...",[impact_dataset/2022/USD0941457-20220118/USD09...,{'FIG. 1': 'FIG. 1 is a top perspective view o...,"{01-01, 11-01}",01,01


In [77]:
sample_df.to_csv("model_io/Impact_Sub.csv", index=False, encoding="utf-8")